# Dia 18 — Gerando gráficos e mapas

**🃏 Cartas:** 18 · 39 · 60 &nbsp;·&nbsp; **📘 Original:** *Day 18. Generating charts and maps*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 18 — Onde estão as vítimas

A Delegada Rita leu o relatório preliminar e devolveu com uma única anotação a caneta, na margem da
tabela de denúncias: *"E no mapa?"*

A Iara ri.

— Toda autoridade é assim. Ninguém lê tabela de 37 linhas. Uma imagem decide em três segundos se o
caso é da delegacia do bairro ou de uma força-tarefa estadual.

Ela abre o mapa de Minas na parede e começa a espetar alfinetes nas cidades que você listou: Belo
Horizonte, Contagem, Uberlândia, Juiz de Fora, Governador Valadares... Na quinta cidade, para.

— Está vendo o desenho? Se o golpe fosse de bairro, os alfinetes estariam todos juntos. Estão
espalhados pelo estado inteiro. Isso é anúncio pago em rede social, grupo de promoção, coisa que
viaja longe. E repare: o endereço do rodapé fica no centro de BH, mas tem vítima a quase 500
quilômetros dali.

Ela devolve a caneta para você.

— Faça três versões: uma para o terminal, que é rápida; uma imagem, para o relatório; e um mapa de
verdade, para a doutora clicar.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-18-graficos-e-mapas")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Fazer um gráfico de barras com `matplotlib` e um mapa com `Basemap`, como no curso original.
- Conhecer uma alternativa leve e interativa ao Basemap (`folium`).
- Desenhar gráficos no **terminal** e em **SVG** só com a biblioteca padrão.
- Gerar um mapa interativo com **Leaflet** a partir do Python.
- Usar área proporcional (raiz quadrada) para não enganar o leitor.

## 🧠 Conceito em 2 minutos

Um gráfico é só uma **regra que transforma número em tamanho**: o maior valor ganha a barra mais
comprida, e os outros, uma fração dela (regra de três). Isso funciona igual no terminal (quantos
`▇`), no SVG (quantos pixels) ou no matplotlib.

**SVG** é uma imagem feita de texto: `<rect x="10" y="20" width="80" height="22"/>` é um retângulo.
Como é texto, o Python escreve SVG com f-strings, e o arquivo abre em qualquer navegador. Cuidado:
no SVG o eixo **y cresce para baixo**.

**Mapa** é um gráfico em que x e y são longitude e latitude. Duas armadilhas:

| Armadilha | Como evitar |
|---|---|
| Ordem das coordenadas | Basemap e GeoJSON usam **(lon, lat)**; folium e Leaflet usam **[lat, lon]** |
| Círculo proporcional | o olho compara **áreas**: raio ∝ √valor (4x vítimas = raio 2x) |

**Leaflet** é uma biblioteca JavaScript de mapas. O Python só escreve o HTML com os dados embutidos
(via `json.dumps`), e o navegador faz o resto, usando os "azulejos" (*tiles*) do OpenStreetMap como
fundo.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/bar.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-18-graficos-e-mapas/original/bar.py) | Gráfico de barras da população de três cidades dos EUA (`matplotlib` + `numpy`) |
| [`original/map.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-18-graficos-e-mapas/original/map.py) | Mapa-múndi com fronteiras, costas e um ponto (`Basemap`) |
| [`original/2026_map.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-18-graficos-e-mapas/original/2026_map.py) | **Alternativa 2026:** o mesmo ponto num mapa interativo com `folium` |

```bash
pip install matplotlib basemap folium
cd dia-18-graficos-e-mapas/original
python bar.py          # gera population_barchart.png
python map.py          # gera map.png (demora um pouco: Basemap é pesado)
python 2026_map.py     # gera map.html (abra no navegador)
```

> ℹ️ **Basemap em 2026.** Ele ainda instala com `pip install basemap` (o pacote traz os dados de
> costas e fronteiras em `basemap_data`) e o `map.py` original continua funcionando. Mas o projeto
> vive em modo de manutenção, e a própria documentação dele aponta o **Cartopy** como sucessor. Para OSINT,
> onde um mapa clicável costuma valer mais que uma imagem, o `folium` é a troca mais simples.

> 🔍 No `map.py`, o ponto `plot(43.00, 39.00)` é **longitude 43, latitude 39** (leste da Turquia).
> Troque a ordem (lon 39, lat 43) e o ponto cai no Mar Negro. É o erro mais comum com mapas.

No Colab, o PNG aparece com `from IPython.display import Image; Image("map.png")`, e o folium aparece
só de escrever o nome da variável do mapa na última linha da célula.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q matplotlib basemap folium

### `original/2026_map.py` — O mesmo mapa com folium (alternativa leve, 2026)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: O mesmo mapa com folium (alternativa leve, 2026)
# ATUALIZAÇÃO 2026 feita para este repositório, BASEADA no original de cipher387
# (Day_18/map.py, MIT © 2023 Cyb_detective). O original continua intacto em map.py
# e ainda funciona (pip install basemap). Esta versão existe porque o Basemap é
# pesado, está em modo de manutenção e gera só imagem parada. O folium gera um mapa
# INTERATIVO em HTML (zoom, arrastar, clicar no ponto), sobre o OpenStreetMap.
# Requer: pip install folium
# ──────────────────────────────────────────────────────────────────────────

# 💬 folium escreve um HTML que usa a biblioteca JavaScript Leaflet.
import folium

# 💬 Mapa-múndi: centro em (lat 20, lon 0) e zoom 2 (bem afastado).
mapa = folium.Map(location=[20, 0], zoom_start=2)

# 💬 O mesmo ponto do original. ATENÇÃO à ordem: o Basemap recebe (longitude, latitude),
# 💬 o folium recebe [latitude, longitude]. Original: plot(43.00, 39.00) = lon 43, lat 39.
folium.CircleMarker(location=[39.00, 43.00], radius=10, color="blue", fill=True,
                    popup="lat 39, lon 43 (leste da Turquia)").add_to(mapa)

# 💬 Grava o HTML. Abra no navegador (ou, no Colab/Jupyter, só escreva "mapa" numa célula).
mapa.save("map.html")
print("map.html gravado")

### `original/bar.py` — Gráfico de barras com Matplotlib

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Gráfico de barras com Matplotlib
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_18/bar.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install matplotlib  (o numpy vem junto)
# ──────────────────────────────────────────────────────────────────────────
# 💬 pyplot é a parte do Matplotlib que desenha gráficos, apelidada de plt.
import matplotlib.pyplot as plt
# 💬 numpy: listas numéricas rápidas (arrays). Aqui só embrulha as listas.
import numpy as np

# 💬 Rótulos das barras (eixo x).
x = np.array(["Los Angeles", "San Francisco", "New York"])
# 💬 Altura de cada barra: população em milhões.
y = np.array([3.8, 0.8, 8.4])

# 💬 Desenha as barras.
plt.bar(x,y)

plt.title ("Population")

# 💬 Grava a imagem PNG na pasta atual.
plt.savefig('population_barchart.png')

### `original/map.py` — Mapa-múndi com Basemap

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Mapa-múndi com Basemap
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_18/map.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install basemap  (pacote pesado; veja a alternativa 2026_map.py)
# ──────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
# 💬 Basemap é um complemento do Matplotlib para mapas. Vive em modo de
# 💬 manutenção; a documentação dele aponta o Cartopy como sucessor.
from mpl_toolkits.basemap import Basemap

# 💬 Uma figura grande: 22 x 22 polegadas.
fig = plt.figure(figsize = (22,22)) 

# 💬 Sem argumentos = mapa-múndi em projeção cilíndrica simples (lon/lat).
map = Basemap()

# 💬 Fronteiras dos países e, abaixo, as linhas de costa.
map.drawcountries()
map.drawcoastlines()

# 💬 Um ponto azul ('b' = blue, 'o' = bolinha). A ordem é LONGITUDE, LATITUDE:
# 💬 43 °E, 39 °N cai na região leste da Turquia. Confundir a ordem é o erro nº 1 com mapas.
map.plot(43.00, 39.00, 'bo', markersize=12)

plt.title("World map", fontsize=20)

# 💬 Grava o mapa como imagem.
plt.savefig('map.png')

## 🇧🇷 Nível 2 — Camada Brasil

Só **biblioteca padrão**, 100% offline para gerar (o mapa Leaflet precisa de internet só na hora de
**abrir**, para baixar o fundo).

| Arquivo | O que faz |
|---|---|
| [`br/01_grafico_terminal.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-18-graficos-e-mapas/br/01_grafico_terminal.py) | Barras `▇` no terminal: vítimas (ou prejuízo) por cidade padronizada |
| [`br/02_grafico_svg.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-18-graficos-e-mapas/br/02_grafico_svg.py) | `saidas/vitimas_por_cidade.svg` desenhado só com strings, com grade e dicas ao passar o mouse |
| [`br/03_mapa_leaflet.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-18-graficos-e-mapas/br/03_mapa_leaflet.py) | `saidas/mapa_vitimas.html` com Leaflet 1.9.4 (unpkg) e círculos de área proporcional |

```bash
cd dia-18-graficos-e-mapas
python br/01_grafico_terminal.py
python br/03_mapa_leaflet.py       # depois abra saidas/mapa_vitimas.html
```

No Colab, os scripts 02 e 03 mostram a imagem e o mapa logo abaixo da célula (com
`IPython.display`, protegido por `try/except ImportError` para não quebrar fora do notebook).

**O que você deve ver no caso:** depois de padronizadas, as 16 grafias viram **11 cidades**. Belo
Horizonte lidera com 8 vítimas, mas a Grande BH (BH, Contagem e Betim) soma só 38% do total. O resto
está espalhado do Triângulo Mineiro ao Vale do Rio Doce.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_grafico_terminal.py` — Gráfico de barras no terminal: vítimas por cidade (já padronizada).

In [ ]:
%cd -q {PASTA_NB}
"""
Gráfico de barras no terminal: vítimas por cidade (já padronizada).

Carta 39 (rank A) — Jutsu do Mapa das Vítimas

O que você aprende:
  • a padronizar cidades digitadas de jeitos diferentes (revisão do Dia 7);
  • a desenhar barras com o caractere ▇, proporcionais ao maior valor;
  • a usar Counter.most_common() para ordenar do maior para o menor.

Como rodar (dentro da pasta dia-18-graficos-e-mapas):
    python br/01_grafico_terminal.py
"""

import csv
import difflib                       # acha a palavra "mais parecida" (corrige 'Belo Horizone')
import unicodedata
from collections import Counter
from decimal import Decimal
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
LARGURA_MAXIMA = 40  #@param {type:"integer"}
MEDIDA = "vitimas"  #@param ["vitimas", "prejuizo"]

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

# Nomes oficiais (como no IBGE) das cidades que aparecem no caso.
OFICIAIS = ["Belo Horizonte", "Betim", "Contagem", "Divinópolis", "Governador Valadares",
            "Ipatinga", "Juiz de Fora", "Poços de Caldas", "Sete Lagoas", "Uberaba", "Uberlândia"]
APELIDOS = {"bh": "Belo Horizonte", "bhz": "Belo Horizonte", "gv": "Governador Valadares", "jf": "Juiz de Fora"}


def chave(texto):
    sem = "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))
    return " ".join(sem.casefold().split())


CHAVES_OFICIAIS = {chave(c): c for c in OFICIAIS}


def padronizar(cidade):
    """'belo horizone' → 'Belo Horizonte' (apelido, depois igualdade, depois o mais parecido)."""
    k = chave(cidade)
    if k in APELIDOS:
        return APELIDOS[k]
    if k in CHAVES_OFICIAIS:
        return CHAVES_OFICIAIS[k]
    parecidas = difflib.get_close_matches(k, CHAVES_OFICIAIS, n=1, cutoff=0.8)
    return CHAVES_OFICIAIS[parecidas[0]] if parecidas else cidade


# ── 1. Contar ─────────────────────────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = list(csv.DictReader(arquivo, delimiter=";"))

contagem = Counter()
for d in denuncias:
    cidade = padronizar(d["cidade"])
    # Counter soma o que você mandar: 1 por vítima, ou o valor em reais.
    contagem[cidade] += 1 if MEDIDA == "vitimas" else Decimal(d["valor"])

# ── 2. Desenhar ───────────────────────────────────────────────────────────
maior = max(contagem.values())
titulo = "Vítimas por cidade" if MEDIDA == "vitimas" else "Prejuízo por cidade (R$)"
print(f"📊 {titulo} — {len(denuncias)} denúncias, {len(contagem)} cidades\n")

for cidade, valor in contagem.most_common():
    # Regra de três: o maior valor ganha LARGURA_MAXIMA blocos, os outros proporcionalmente.
    blocos = round(valor / maior * LARGURA_MAXIMA)
    rotulo = f"{valor}" if MEDIDA == "vitimas" else f"{valor:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")
    # :<22 = alinha à esquerda em 22 colunas; max(blocos, 1) garante pelo menos um bloquinho
    print(f"{cidade:<22} {'▇' * max(blocos, 1)} {rotulo}")

# ── 3. Uma leitura analítica ──────────────────────────────────────────────
rmbh = {"Belo Horizonte", "Contagem", "Betim"}       # Região Metropolitana de BH (parte dela)
na_rmbh = sum(v for c, v in contagem.items() if c in rmbh)
print(f"\n🧭 Grande BH (BH + Contagem + Betim): {na_rmbh} de {sum(contagem.values())} "
      f"({na_rmbh / sum(contagem.values()):.0%}). O resto está espalhado pelo interior de MG.")
print("💡 Golpe espalhado por várias regiões sugere divulgação online ampla (anúncio pago, grupos), não boca a boca local.")

### `br/02_grafico_svg.py` — Gráfico de barras em SVG gerado só com strings (sem matplotlib).

In [ ]:
%cd -q {PASTA_NB}
"""
Gráfico de barras em SVG gerado só com strings (sem matplotlib).

Carta 39 (rank A) — Jutsu do Mapa das Vítimas (parte 2)

O que você aprende:
  • que SVG é uma imagem feita de TEXTO (retângulos, linhas, textos com coordenadas);
  • a converter números em pixels (escala) e a lembrar que o eixo y do SVG cresce para BAIXO;
  • a gravar um arquivo .svg que abre no navegador, no Word e no GitHub.

Como rodar (dentro da pasta dia-18-graficos-e-mapas):
    python br/02_grafico_svg.py
    (depois abra saidas/vitimas_por_cidade.svg no navegador)
"""

import csv
import difflib
import html
import unicodedata
from collections import Counter
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
TITULO = "Operação Vitrine Fantasma — vítimas por cidade"  #@param {type:"string"}
COR_BARRA = "#1f5fa8"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

OFICIAIS = ["Belo Horizonte", "Betim", "Contagem", "Divinópolis", "Governador Valadares",
            "Ipatinga", "Juiz de Fora", "Poços de Caldas", "Sete Lagoas", "Uberaba", "Uberlândia"]


def chave(texto):
    sem = "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))
    return " ".join(sem.casefold().split())


CHAVES_OFICIAIS = {chave(c): c for c in OFICIAIS}


def padronizar(cidade):
    k = chave(cidade)
    if k in CHAVES_OFICIAIS:
        return CHAVES_OFICIAIS[k]
    parecidas = difflib.get_close_matches(k, CHAVES_OFICIAIS, n=1, cutoff=0.8)
    return CHAVES_OFICIAIS[parecidas[0]] if parecidas else cidade


# ── 1. Dados ──────────────────────────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    contagem = Counter(padronizar(d["cidade"]) for d in csv.DictReader(arquivo, delimiter=";"))
itens = contagem.most_common()                 # [("Belo Horizonte", 8), ...]

# ── 2. Geometria (tudo em pixels) ─────────────────────────────────────────
# Barras HORIZONTAIS: nomes de cidade longos cabem melhor à esquerda.
MARGEM_ESQ, MARGEM_TOPO = 170, 60              # espaço para os nomes e para o título
ALTURA_BARRA, ESPACO = 22, 8
LARGURA_UTIL = 420                             # largura da barra mais comprida
largura = MARGEM_ESQ + LARGURA_UTIL + 60
altura = MARGEM_TOPO + len(itens) * (ALTURA_BARRA + ESPACO) + 40
maior = itens[0][1]


def escala(valor):
    """Converte 'número de vítimas' em 'pixels de largura'."""
    return valor / maior * LARGURA_UTIL


# ── 3. Desenhar: cada elemento é uma linha de texto ───────────────────────
partes = [
    # viewBox define o sistema de coordenadas; width/height o tamanho na tela.
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{largura}" height="{altura}" '
    f'viewBox="0 0 {largura} {altura}" font-family="system-ui, sans-serif">',
    f'<rect width="{largura}" height="{altura}" fill="white"/>',
    f'<text x="20" y="32" font-size="18" font-weight="bold" fill="#1d2433">{html.escape(TITULO)}</text>',
]

# Linhas de grade verticais a cada 2 vítimas (ajudam a ler o valor)
for marca in range(0, maior + 1, 2):
    x = MARGEM_ESQ + escala(marca)
    partes.append(f'<line x1="{x:.1f}" y1="{MARGEM_TOPO - 8}" x2="{x:.1f}" y2="{altura - 30}" stroke="#e3e7ee"/>')
    partes.append(f'<text x="{x:.1f}" y="{altura - 12}" font-size="11" fill="#5b6475" text-anchor="middle">{marca}</text>')

for i, (cidade, valor) in enumerate(itens):
    # No SVG, y=0 é o TOPO. Cada barra desce (ALTURA_BARRA + ESPACO) pixels em relação à anterior.
    y = MARGEM_TOPO + i * (ALTURA_BARRA + ESPACO)
    meio = y + ALTURA_BARRA / 2 + 4             # +4: centraliza o texto na barra visualmente
    partes.append(f'<text x="{MARGEM_ESQ - 10}" y="{meio:.1f}" font-size="13" text-anchor="end" '
                  f'fill="#1d2433">{html.escape(cidade)}</text>')
    partes.append(f'<rect x="{MARGEM_ESQ}" y="{y}" width="{escala(valor):.1f}" height="{ALTURA_BARRA}" '
                  f'rx="3" fill="{html.escape(COR_BARRA)}"><title>{html.escape(cidade)}: {valor}</title></rect>')
    # <title> dentro do retângulo vira "dica" quando você passa o mouse por cima.
    partes.append(f'<text x="{MARGEM_ESQ + escala(valor) + 6:.1f}" y="{meio:.1f}" font-size="12" '
                  f'fill="#1d2433">{valor}</text>')

partes.append("</svg>")
svg = "\n".join(partes)

SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "vitimas_por_cidade.svg"
destino.write_text(svg, encoding="utf-8")
print(f"🖼️ {destino.relative_to(PASTA_DIA)} gravado: {len(itens)} barras, {largura}x{altura} px, {len(svg)} caracteres.")
print("💡 Abra no navegador. Dá para colar no relatório HTML do Dia 17 (SVG é texto, cabe dentro do HTML).")

# No Colab/Jupyter, mostra a imagem abaixo da célula.
if "__file__" not in globals():
    try:
        from IPython.display import SVG, display
        display(SVG(svg))
    except ImportError:
        pass

### `br/03_mapa_leaflet.py` — Mapa interativo das vítimas com Leaflet, gerado só com strings.

In [ ]:
%cd -q {PASTA_NB}
"""
Mapa interativo das vítimas com Leaflet, gerado só com strings.

Carta 39 (rank A) — Jutsu do Mapa das Vítimas (parte 3)

O que você aprende:
  • a escrever uma página HTML que carrega o Leaflet (biblioteca de mapas em JavaScript) por CDN;
  • a passar dados do Python para o JavaScript com json.dumps (sem montar texto na mão);
  • a desenhar círculos com área proporcional ao número de vítimas (raiz quadrada!).

As coordenadas são APROXIMADAS (centro de cada município), embutidas no script: suficiente para
um mapa de distribuição, nunca para localizar uma pessoa.

Como rodar (dentro da pasta dia-18-graficos-e-mapas):
    python br/03_mapa_leaflet.py
    (abra saidas/mapa_vitimas.html no navegador: precisa de internet para baixar o mapa de fundo)
"""

import csv
import difflib
import json
import math
import unicodedata
from collections import Counter, defaultdict
from decimal import Decimal
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
RAIO_MAXIMO_KM = 30  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

# Centro aproximado de cada município (latitude, longitude), arredondado.
COORDENADAS = {
    "Belo Horizonte": (-19.92, -43.94),
    "Betim": (-19.97, -44.20),
    "Contagem": (-19.93, -44.05),
    "Divinópolis": (-20.14, -44.89),
    "Governador Valadares": (-18.85, -41.95),
    "Ipatinga": (-19.47, -42.54),
    "Juiz de Fora": (-21.76, -43.35),
    "Poços de Caldas": (-21.79, -46.56),
    "Sete Lagoas": (-19.47, -44.25),
    "Uberaba": (-19.75, -47.93),
    "Uberlândia": (-18.92, -48.28),
}


def chave(texto):
    sem = "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))
    return " ".join(sem.casefold().split())


CHAVES_OFICIAIS = {chave(c): c for c in COORDENADAS}


def padronizar(cidade):
    k = chave(cidade)
    if k in CHAVES_OFICIAIS:
        return CHAVES_OFICIAIS[k]
    parecidas = difflib.get_close_matches(k, CHAVES_OFICIAIS, n=1, cutoff=0.8)
    return CHAVES_OFICIAIS[parecidas[0]] if parecidas else cidade


# ── 1. Agregar por cidade ─────────────────────────────────────────────────
vitimas = Counter()
prejuizo = defaultdict(Decimal)
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    for d in csv.DictReader(arquivo, delimiter=";"):
        cidade = padronizar(d["cidade"])
        vitimas[cidade] += 1
        prejuizo[cidade] += Decimal(d["valor"])

sem_coordenada = [c for c in vitimas if c not in COORDENADAS]
if sem_coordenada:
    print(f"⚠️ Sem coordenada (ficam fora do mapa): {sem_coordenada}")

# ── 2. Tamanho do círculo ─────────────────────────────────────────────────
# Nosso olho compara ÁREAS, não raios. Se uma cidade tem 4x mais vítimas, o círculo dela
# deve ter 4x mais área, ou seja, raio 2x maior. Por isso usamos a raiz quadrada.
maior = max(vitimas.values())
pontos = []
for cidade, qtd in vitimas.most_common():
    if cidade not in COORDENADAS:
        continue
    lat, lon = COORDENADAS[cidade]
    raio_m = RAIO_MAXIMO_KM * 1000 * math.sqrt(qtd / maior)
    total = f"{prejuizo[cidade]:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")
    pontos.append({"cidade": cidade, "lat": lat, "lon": lon, "vitimas": qtd,
                   "raio": round(raio_m), "prejuizo": f"R$ {total}"})
    print(f"📍 {cidade:<22} {qtd:>2} vítimas · raio {raio_m / 1000:5.1f} km · R$ {total}")

# ── 3. A página ───────────────────────────────────────────────────────────
# json.dumps transforma a lista Python em texto JavaScript válido (aspas, acentos, tudo certo).
# Leaflet 1.9.4 via unpkg, com versão fixa (nunca "latest": a página não muda sozinha amanhã).
pagina = """<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Mapa das vítimas — Operação Vitrine Fantasma</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>html, body, #mapa { height: 100%; margin: 0; } .caixa { font: 14px system-ui; }</style>
</head>
<body>
<div id="mapa"></div>
<script>
const pontos = __PONTOS__;
const mapa = L.map("mapa").setView([-19.9, -44.5], 7);
L.tileLayer("https://tile.openstreetmap.org/{z}/{x}/{y}.png", {
  maxZoom: 18,
  attribution: "&copy; colaboradores do OpenStreetMap"
}).addTo(mapa);
for (const p of pontos) {
  L.circle([p.lat, p.lon], {radius: p.raio, color: "#b3261e", fillOpacity: 0.35})
    .bindPopup(`<div class="caixa"><b>${p.cidade}</b><br>${p.vitimas} vítima(s)<br>${p.prejuizo}</div>`)
    .addTo(mapa);
}
const grupo = L.featureGroup(pontos.map(p => L.marker([p.lat, p.lon])));
mapa.fitBounds(grupo.getBounds().pad(0.2));
</script>
</body>
</html>
""".replace("__PONTOS__", json.dumps(pontos, ensure_ascii=False))
# Usamos .replace em vez de f-string porque o JavaScript está cheio de chaves { }.
# Os dados (nomes de cidade) vêm do NOSSO dicionário, não de terceiros. Se viessem
# de fora, seria preciso escapar o HTML no popup (ver Dia 20, XSS).

SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "mapa_vitimas.html"
destino.write_text(pagina, encoding="utf-8")
print(f"\n🗺️ {destino.relative_to(PASTA_DIA)} gravado com {len(pontos)} cidades.")
print("💡 Abra no navegador (precisa de internet para o mapa de fundo e o Leaflet).")

# No Colab/Jupyter, mostra o mapa abaixo da célula.
if "__file__" not in globals():
    try:
        from IPython.display import HTML, display
        display(HTML(f'<iframe srcdoc="{pagina.replace(chr(34), "&quot;")}" width="100%" height="500"></iframe>'))
    except ImportError:
        pass

## ⚖️ Ética, LGPD e OPSEC

- **Agregue antes de mapear.** O mapa mostra **cidades**, nunca endereços de vítimas. Ponto exato de
  pessoa num mapa é dado pessoal sensível na prática (permite achar a casa de alguém).
- **Cuidado com cidades pequenas.** "1 vítima em uma cidade de 3 mil habitantes" pode identificar a
  pessoa. Em relatórios públicos, agrupe por região quando a contagem for muito baixa.
- **Mapa online vaza.** O HTML com Leaflet baixa azulejos do OpenStreetMap: o servidor do mapa vê seu
  IP e a região que você está olhando. Para casos sensíveis, prefira o mapa offline do desafio.
- **Gráfico também mente.** Barra que não começa no zero, círculo com raio (e não área)
  proporcional, cores alarmistas: tudo isso distorce. Relatório de investigação tem que ser sóbrio.
- Os azulejos do OpenStreetMap têm política de uso: atribuição obrigatória e nada de baixar em massa.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. No Basemap, map.plot(43.00, 39.00) marca qual ponto?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Latitude 43, longitude 39", "B) Longitude 43, latitude 39", "C) O pixel (43, 39) da imagem", "D) Depende do zoom"]
correta = "B) Longitude 43, latitude 39"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Basemap (como o GeoJSON) recebe x = longitude e y = latitude. folium e Leaflet usam a ordem [lat, lon].")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Uma cidade tem 4 vezes mais vítimas que outra. O raio do círculo dela deve ser…
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) 4 vezes maior", "B) 2 vezes maior", "C) 16 vezes maior", "D) igual, só a cor muda"]
correta = "B) 2 vezes maior"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O olho compara áreas; área ∝ raio², então o raio deve ser proporcional à raiz quadrada do valor.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Por que o script do mapa Leaflet usa json.dumps para colocar os dados no HTML?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Para deixar o arquivo menor", "B) Porque o Leaflet só lê arquivos .json", "C) Para gerar JavaScript válido (aspas, acentos e listas) sem montar o texto na mão", "D) Para criptografar os dados"]
correta = "C) Para gerar JavaScript válido (aspas, acentos e listas) sem montar o texto na mão"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Uma lista de dicionários Python vira um array de objetos JavaScript válido em uma linha.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 18: Cartógrafo offline

**Carta 60 · Rank S**

## A missão

A Iara precisa levar o mapa para uma reunião numa sala sem internet. E a Delegada Rita fez uma
pergunta nova: **a que distância as vítimas estão do "endereço" da loja?**

Escreva um script que:

1. Padronize as cidades de `denuncias.csv` e conte as vítimas por cidade.
2. Converta latitude/longitude em x/y de um SVG (projeção **equirretangular**: x vem da longitude,
   y da latitude), com a longitude multiplicada pelo **cosseno da latitude média** para o mapa não
   ficar "esticado".
3. Desenhe um círculo por cidade com **área** proporcional às vítimas, com o nome ao lado, e um
   ponto preto no endereço do rodapé (centro de BH).
4. Calcule a distância em linha reta de cada cidade até o rodapé com a **fórmula de haversine**.
5. Imprima as cidades da mais perto para a mais longe e diga quantas vítimas estão a mais de 200 km.
6. Grave `saidas/mapa_offline.svg`, que abre **sem internet**.

## Critérios de rank S

- [ ] Só biblioteca padrão (`math`, `csv`, `html`...), sem azulejos online
- [ ] Norte para cima: Uberlândia à esquerda, Governador Valadares à direita, Juiz de Fora embaixo
- [ ] Raio ∝ √vítimas
- [ ] Distância BH → Uberlândia entre 450 e 500 km

## Dica

Haversine: `h = sen²(Δlat/2) + cos(lat1)·cos(lat2)·sen²(Δlon/2)` e `d = 2·R·asen(√h)`, com
R = 6371 km e todos os ângulos em **radianos** (`math.radians`).

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 18 — Mapa offline + distâncias.

Desenha um mapa das vítimas em SVG SEM internet e SEM biblioteca de mapas: converte
latitude/longitude em x/y (projeção equirretangular com correção pelo cosseno da latitude),
com círculos de área proporcional às vítimas. Calcula a distância (fórmula de haversine)
de cada cidade até o endereço do rodapé da loja (CEP 30130-010, centro de BH).
"""

import csv
import difflib
import html
import math
import unicodedata
from collections import Counter
from pathlib import Path

LARGURA_PX = 640  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

COORDENADAS = {
    "Belo Horizonte": (-19.92, -43.94), "Betim": (-19.97, -44.20), "Contagem": (-19.93, -44.05),
    "Divinópolis": (-20.14, -44.89), "Governador Valadares": (-18.85, -41.95),
    "Ipatinga": (-19.47, -42.54), "Juiz de Fora": (-21.76, -43.35), "Poços de Caldas": (-21.79, -46.56),
    "Sete Lagoas": (-19.47, -44.25), "Uberaba": (-19.75, -47.93), "Uberlândia": (-18.92, -48.28),
}
# Centro de BH, região do CEP 30130-010 do rodapé (aproximado; ver Dia 5)
RODAPE = (-19.92, -43.94)


def chave(t):
    s = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
    return " ".join(s.casefold().split())


OFICIAIS = {chave(c): c for c in COORDENADAS}


def padronizar(cidade):
    k = chave(cidade)
    if k in OFICIAIS:
        return OFICIAIS[k]
    p = difflib.get_close_matches(k, OFICIAIS, n=1, cutoff=0.8)
    return OFICIAIS[p[0]] if p else cidade


def haversine_km(a, b):
    """Distância em linha reta sobre a esfera terrestre (raio médio 6371 km)."""
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))


with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as f:
    vitimas = Counter(padronizar(d["cidade"]) for d in csv.DictReader(f, delimiter=";"))

# ── Projeção: lon → x, lat → y ────────────────────────────────────────────
lats = [c[0] for c in COORDENADAS.values()]
lons = [c[1] for c in COORDENADAS.values()]
margem = 0.8                                        # graus de folga em volta
lat_min, lat_max = min(lats) - margem, max(lats) + margem
lon_min, lon_max = min(lons) - margem, max(lons) + margem
# Perto do Equador 1° de longitude ≈ 1° de latitude; em -20° ele "encolhe" pelo cosseno.
fator = math.cos(math.radians((lat_min + lat_max) / 2))
escala = LARGURA_PX / ((lon_max - lon_min) * fator)
altura_px = round((lat_max - lat_min) * escala)


def xy(lat, lon):
    x = (lon - lon_min) * fator * escala
    y = (lat_max - lat) * escala                    # y do SVG cresce para baixo: invertemos
    return x, y


maior = max(vitimas.values())
svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{LARGURA_PX}" height="{altura_px + 30}" '
       f'font-family="system-ui, sans-serif">',
       f'<rect width="100%" height="100%" fill="#f4f7fb"/>',
       f'<text x="10" y="{altura_px + 20}" font-size="11" fill="#5b6475">'
       f'Projeção equirretangular · coordenadas aproximadas · círculo = área ∝ vítimas</text>']
xr, yr = xy(*RODAPE)
svg.append(f'<circle cx="{xr:.1f}" cy="{yr:.1f}" r="4" fill="black"><title>CEP do rodapé</title></circle>')

linhas = []
for cidade, qtd in vitimas.most_common():
    lat, lon = COORDENADAS[cidade]
    x, y = xy(lat, lon)
    r = 22 * math.sqrt(qtd / maior)
    dist = haversine_km(RODAPE, (lat, lon))
    linhas.append((dist, cidade, qtd))
    svg.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" fill="#b3261e" fill-opacity=".35" stroke="#b3261e">'
               f'<title>{html.escape(cidade)}: {qtd}</title></circle>')
    svg.append(f'<text x="{x + r + 3:.1f}" y="{y + 4:.1f}" font-size="12">{html.escape(cidade)} ({qtd})</text>')
svg.append("</svg>")

SAIDAS.mkdir(exist_ok=True)
(SAIDAS / "mapa_offline.svg").write_text("\n".join(svg), encoding="utf-8")

print("📏 Distância em linha reta até o endereço do rodapé (centro de BH):\n")
for dist, cidade, qtd in sorted(linhas):
    print(f"   {cidade:<22} {dist:6.0f} km  {'●' * qtd}")
longe = sum(q for d, _, q in linhas if d > 200)
print(f"\n🧭 {longe} de {sum(vitimas.values())} vítimas moram a mais de 200 km do 'endereço' da loja.")
print("   Ninguém iria buscar a compra lá: o endereço só existia para parecer confiável.")
print("💾 saidas/mapa_offline.svg gravado (abre sem internet).")

## 🔎 Inovação OSINT do dia

**Overture Maps: base aberta de lugares do mundo, com cliente Python.** A Overture Maps Foundation
(Linux Foundation, com Amazon, Meta, Microsoft e TomTom) publica dados abertos de lugares
(estabelecimentos), prédios, endereços e vias. O cliente `pip install overturemaps` baixa só uma
região: `overturemaps download --bbox=oeste,sul,leste,norte -f geojson --type=place -o lugares.geojson`
(atenção à ordem: longitude, latitude). Em OSINT, serve para checar se um "endereço comercial" de uma
loja suspeita corresponde a algum estabelecimento conhecido naquele quarteirão, sem raspar nenhum
site. O GeoJSON resultante abre direto no folium ou no Leaflet.

## 📚 Para ir além

- Leaflet (tutoriais em inglês, muito visuais): <https://leafletjs.com/examples.html>
- folium: <https://python-visualization.github.io/folium/>
- Basemap (status e instalação): <https://matplotlib.org/basemap/>
- Cartopy: <https://scitools.org.uk/cartopy/>
- Política de uso dos azulejos do OpenStreetMap: <https://operations.osmfoundation.org/policies/tiles/>
- Cliente Python do Overture Maps: <https://docs.overturemaps.org/getting-data/overturemaps-py/>
- Malhas municipais oficiais do IBGE (para mapas com contorno das cidades): <https://www.ibge.gov.br/geociencias/organizacao-do-territorio/malhas-territoriais.html>

## ✅ Checklist

- [ ] Rodei `bar.py` e `map.py` (ou o `2026_map.py`)
- [ ] Entendi a diferença entre (lon, lat) e [lat, lon]
- [ ] Vi o gráfico de barras no terminal
- [ ] Abri o SVG no navegador e passei o mouse nas barras
- [ ] Abri o mapa Leaflet e cliquei nas cidades
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 18](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)